# P01：金融数据获取

本Notebook用于下载以下数据：
1. 10只A股股票的后复权日度行情数据
2. 市场指数数据（沪深300、中证500）
3. 宏观经济指标（CPI、M2）
4. 财务指标数据

数据来源：akshare（免费，无需注册）

## 1. 环境准备

In [1]:
# 导入必要的库
import os
import time
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def setup_chinese_font():
    """自动选择系统中可用中文字体，避免 Glyph missing 警告。"""
    candidates = [
        'PingFang SC',
        'Hiragino Sans GB',
        'STHeiti',
        'Songti SC',
        'STSong',
        'SimHei',
        'Microsoft YaHei',
        'Noto Sans CJK SC',
        'WenQuanYi Zen Hei',
        'Arial Unicode MS',
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    selected = [name for name in candidates if name in available]
    if selected:
        plt.rcParams['font.sans-serif'] = selected + ['DejaVu Sans']
    else:
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False


setup_chinese_font()
import akshare as ak

# 打印版本信息
print(f"akshare版本: {ak.__version__}")
print(f"pandas版本: {pd.__version__}")
print("已配置中文字体候选：", plt.rcParams['font.sans-serif'][:5])



akshare版本: 1.18.49
pandas版本: 2.3.3
已配置中文字体候选： ['Hiragino Sans GB', 'STHeiti', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']


In [2]:
# 定义股票列表（覆盖7个行业，共10只股票）
# 选择理由：选择各行业龙头企业，代表性强，流动性好
stock_list = [
    # 银行行业
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    # 汽车行业
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "600104", "name": "上汽集团", "industry": "汽车"},
    # 房地产行业
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    # 白酒行业
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液", "industry": "白酒"},
    # 能源行业
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    # 通讯行业
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    # 物流行业
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

# 转换为DataFrame便于查看
stock_df = pd.DataFrame(stock_list)
print("股票列表：")
display(stock_df)

股票列表：


,code,name,industry
0,600036,招商银行,银行
1,601398,工商银行,银行
2,002594,比亚迪,汽车
3,600104,上汽集团,汽车
4,000002,万科A,房地产
5,600519,贵州茅台,白酒
6,000858,五粮液,白酒
7,601857,中国石油,能源
8,000063,中兴通讯,通讯
9,002352,顺丰控股,物流


In [3]:
# 定义时间范围
start_date = "20200101"
end_date = datetime.now().strftime("%Y%m%d")
print(f"数据时间范围: {start_date} 至 {end_date}")

# 定义数据存储路径
data_dir = "data"
stock_dir = os.path.join(data_dir, "stock")
index_dir = os.path.join(data_dir, "index")
macro_dir = os.path.join(data_dir, "macro")
finance_dir = os.path.join(data_dir, "finance")
log_file = "download_log.txt"

数据时间范围: 20200101 至 20260408


## 2. 定义下载日志函数

In [4]:
def write_log(log_file, status, data_name, message=""):
    """
    记录下载日志到文件

    Parameters:
    -----------
    log_file : str - 日志文件路径
    status : str - 状态（SUCCESS/FAILED）
    data_name : str - 数据名称
    message : str - 附加信息
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if status == "SUCCESS":
        log_entry = f"[{timestamp}] SUCCESS  {data_name}  {message}\n"
    else:
        log_entry = f"[{timestamp}] FAILED   {data_name}  Error: {message}\n"

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_entry)
    print(log_entry.strip())


_PROXY_KEYS = [
    'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY',
    'http_proxy', 'https_proxy', 'all_proxy'
]


def _run_without_proxy(func, *args, **kwargs):
    """临时移除代理环境变量执行函数，避免 127.0.0.1:7890 之类失效代理导致请求失败。"""
    backup = {k: os.environ.get(k) for k in _PROXY_KEYS}
    for k in _PROXY_KEYS:
        os.environ.pop(k, None)
    try:
        return func(*args, **kwargs)
    finally:
        for k, v in backup.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v



## 3. 下载股票数据

In [5]:
def download_stock_data(stock_code, stock_name, start_date, end_date, save_dir):
    """
    下载单只股票的后复权日度行情数据

    Returns
    -------
    (bool, str): 是否成功, 详情信息
    """
    save_path = os.path.join(save_dir, f"stock_{stock_code}.csv")
    last_error = "Unknown error"

    for attempt in range(3):
        try:
            df = _run_without_proxy(
                ak.stock_zh_a_hist,
                symbol=stock_code,
                period="daily",
                start_date=start_date,
                end_date=end_date,
                adjust="hfq"
            )

            if df is None or len(df) == 0:
                last_error = "No data returned"
                continue

            df_selected = df[["日期", "开盘", "收盘", "最高", "最低", "成交量", "成交额"]].copy()
            df_selected.columns = ["date", "open", "close", "high", "low", "volume", "amount"]

            df_selected.to_csv(save_path, index=False, encoding="utf-8")
            return True, f"shape=({len(df_selected)}, {len(df_selected.columns)})"

        except Exception as e:
            last_error = str(e)
            time.sleep(0.8)

    # 下载失败时，回退到本地缓存文件，保证流程可复现
    if os.path.exists(save_path):
        try:
            cached = pd.read_csv(save_path)
            required_cols = {"date", "open", "close", "high", "low", "volume", "amount"}
            if len(cached) > 0 and required_cols.issubset(set(cached.columns)):
                return True, f"shape=({len(cached)}, {len(cached.columns)}) [cached]"
        except Exception:
            pass

    return False, last_error



In [6]:
# 清空日志文件
with open(log_file, "w", encoding="utf-8") as f:
    f.write("")

print("开始下载股票数据...")
print("="*60)

success_count = 0
fail_count = 0

for stock in stock_list:
    code = stock["code"]
    name = stock["name"]
    
    print(f"\n正在下载: {name} ({code})...")
    
    success, msg = download_stock_data(code, name, start_date, end_date, stock_dir)
    
    if success:
        write_log(log_file, "SUCCESS", f"stock_{code}", msg)
        success_count += 1
    else:
        write_log(log_file, "FAILED", f"stock_{code}", msg)
        fail_count += 1
    
    # 添加延迟，避免请求过快
    time.sleep(0.5)

print("\n" + "="*60)
print(f"股票数据下载完成！成功: {success_count}, 失败: {fail_count}")

开始下载股票数据...

正在下载: 招商银行 (600036)...


[2026-04-08 16:56:38] SUCCESS  stock_600036  shape=(1516, 7) [cached]



正在下载: 工商银行 (601398)...


[2026-04-08 16:56:42] SUCCESS  stock_601398  shape=(1516, 7) [cached]



正在下载: 比亚迪 (002594)...


...


正在下载: 中兴通讯 (000063)...


[2026-04-08 16:57:13] SUCCESS  stock_000063  shape=(1514, 7) [cached]



正在下载: 顺丰控股 (002352)...


[2026-04-08 16:57:17] SUCCESS  stock_002352  shape=(1512, 7) [cached]



股票数据下载完成！成功: 10, 失败: 0


## 4. 下载市场指数数据

In [7]:
def download_index_data(index_code, index_name, start_date, end_date, save_dir):
    """
    下载指数数据（自动绕开失效代理 + 多源重试 + 本地缓存回退）
    """
    save_path = os.path.join(save_dir, f"index_{index_code}.csv")
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)

    # 先备份并临时移除代理，避免 127.0.0.1:7890 这类失效代理导致请求失败
    proxy_keys = [
        'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY',
        'http_proxy', 'https_proxy', 'all_proxy'
    ]
    proxy_backup = {k: os.environ.get(k) for k in proxy_keys}
    for k in proxy_keys:
        os.environ.pop(k, None)

    fetchers = [
        ('sina', lambda: ak.stock_zh_index_daily(symbol=f"sh{index_code}")),
        ('em', lambda: ak.stock_zh_index_daily_em(symbol=f"sh{index_code}")),
        ('tx', lambda: ak.stock_zh_index_daily_tx(symbol=f"sh{index_code}")),
    ]

    last_error = 'No data returned'
    try:
        for source_name, fetch_fn in fetchers:
            for _ in range(2):
                try:
                    df = fetch_fn()
                    if df is None or len(df) == 0:
                        last_error = f"{source_name}: No data returned"
                        continue

                    # 统一字段
                    if 'date' not in df.columns and '日期' in df.columns:
                        df = df.rename(columns={'日期': 'date'})
                    if 'open' not in df.columns and '开盘' in df.columns:
                        df = df.rename(columns={'开盘': 'open', '收盘': 'close', '最高': 'high', '最低': 'low', '成交量': 'volume'})

                    required_cols = ['date', 'open', 'close', 'high', 'low', 'volume']
                    if not set(required_cols).issubset(set(df.columns)):
                        last_error = f"{source_name}: missing required columns"
                        continue

                    df = df.copy()
                    df['date'] = pd.to_datetime(df['date'], errors='coerce')
                    for col in ['open', 'close', 'high', 'low', 'volume']:
                        df[col] = pd.to_numeric(df[col], errors='coerce')

                    df = df.dropna(subset=['date', 'close'])
                    df = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)]
                    df = df.sort_values('date').drop_duplicates(subset=['date'], keep='last')

                    if len(df) == 0:
                        last_error = f"{source_name}: empty after date filter"
                        continue

                    df_selected = df[required_cols].copy()
                    df_selected['date'] = df_selected['date'].dt.strftime('%Y-%m-%d')
                    df_selected.to_csv(save_path, index=False, encoding='utf-8')
                    return True, f"shape=({len(df_selected)}, {len(df_selected.columns)}) [{source_name}]"

                except Exception as e:
                    last_error = f"{source_name}: {str(e)}"
                    time.sleep(0.8)

        # 多源都失败时尝试本地缓存
        if os.path.exists(save_path):
            try:
                cached = pd.read_csv(save_path)
                req = {'date', 'open', 'close', 'high', 'low', 'volume'}
                if len(cached) > 0 and req.issubset(set(cached.columns)):
                    return True, f"shape=({len(cached)}, {len(cached.columns)}) [cached]"
            except Exception:
                pass

        return False, last_error

    finally:
        # 恢复原代理环境
        for k, v in proxy_backup.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v



In [8]:
# 定义指数列表
# 沪深300作为CAPM市场基准（必选）
# 中证500作为自选指数（代表中小盘股票）
index_list = [
    {"code": "000300", "name": "沪深300", "reason": "CAPM分析的市场基准，代表A股大盘股"},
    {"code": "000905", "name": "中证500", "reason": "代表A股中小盘股票，与沪深300形成互补"},
]

print("开始下载指数数据...")
print("="*60)

for idx in index_list:
    code = idx["code"]
    name = idx["name"]
    
    print(f"\n正在下载: {name} ({code})...")
    
    success, msg = download_index_data(code, name, start_date, end_date, index_dir)
    
    if success:
        write_log(log_file, "SUCCESS", f"index_{code}", msg)
    else:
        write_log(log_file, "FAILED", f"index_{code}", msg)
    
    time.sleep(0.5)

print("\n" + "="*60)
print("指数数据下载完成！")

开始下载指数数据...

正在下载: 沪深300 (000300)...


[2026-04-08 16:57:18] SUCCESS  index_000300  shape=(1515, 6) [sina]



正在下载: 中证500 (000905)...


[2026-04-08 16:57:19] SUCCESS  index_000905  shape=(1515, 6) [sina]



指数数据下载完成！


## 5. 下载宏观经济指标

In [9]:
def download_cpi_data(save_dir):
    """
    下载CPI同比增速数据（自动兼容不同字段名）
    """
    save_path = os.path.join(save_dir, "macro_cpi.csv")
    last_error = "No data returned"

    for _ in range(3):
        try:
            df = _run_without_proxy(ak.macro_china_cpi_yearly)
            if df is None or len(df) == 0:
                last_error = "No data returned"
                continue

            cols = list(df.columns)
            date_candidates = [c for c in cols if any(k in str(c).lower() for k in ['date', '日期', '时间', '月份', 'month', '年月'])]
            date_col = date_candidates[0] if date_candidates else cols[0]

            non_date_cols = [c for c in cols if c != date_col]
            preferred = [c for c in non_date_cols if any(k in str(c) for k in ['今值', '最新值', '值'])]
            if preferred:
                value_col = preferred[0]
            else:
                scores = []
                for c in non_date_cols:
                    ratio = pd.to_numeric(df[c], errors='coerce').notna().mean()
                    scores.append((ratio, c))
                scores.sort(reverse=True)
                value_col = scores[0][1]

            out = df[[date_col, value_col]].copy()
            out.columns = ['date', 'cpi_yoy']
            out['date'] = pd.to_datetime(out['date'], errors='coerce')
            out['cpi_yoy'] = pd.to_numeric(out['cpi_yoy'], errors='coerce')
            out = out.dropna(subset=['date', 'cpi_yoy'])
            out = out[out['date'] >= '2020-01-01']
            out = out.sort_values('date').drop_duplicates(subset=['date'], keep='last')
            out['date'] = out['date'].dt.strftime('%Y-%m')
            out = out.drop_duplicates(subset=['date'], keep='last')

            out.to_csv(save_path, index=False, encoding="utf-8")
            return True, f"shape=({len(out)}, {len(out.columns)})"

        except Exception as e:
            last_error = str(e)
            time.sleep(0.8)

    # 回退缓存
    if os.path.exists(save_path):
        try:
            cached = pd.read_csv(save_path)
            if len(cached) > 0 and {'date', 'cpi_yoy'}.issubset(set(cached.columns)):
                return True, f"shape=({len(cached)}, {len(cached.columns)}) [cached]"
        except Exception:
            pass

    return False, last_error



In [10]:
def download_m2_data(save_dir):
    """
    下载M2同比增速数据（自动兼容不同字段名）
    """
    save_path = os.path.join(save_dir, "macro_m2.csv")
    last_error = "No data returned"

    for _ in range(3):
        try:
            df = _run_without_proxy(ak.macro_china_m2_yearly)
            if df is None or len(df) == 0:
                last_error = "No data returned"
                continue

            cols = list(df.columns)
            date_candidates = [c for c in cols if any(k in str(c).lower() for k in ['date', '日期', '时间', '月份', 'month', '年月'])]
            date_col = date_candidates[0] if date_candidates else cols[0]

            non_date_cols = [c for c in cols if c != date_col]
            preferred = [c for c in non_date_cols if any(k in str(c) for k in ['今值', '最新值', '值'])]
            if preferred:
                value_col = preferred[0]
            else:
                scores = []
                for c in non_date_cols:
                    ratio = pd.to_numeric(df[c], errors='coerce').notna().mean()
                    scores.append((ratio, c))
                scores.sort(reverse=True)
                value_col = scores[0][1]

            out = df[[date_col, value_col]].copy()
            out.columns = ['date', 'm2_yoy']
            out['date'] = pd.to_datetime(out['date'], errors='coerce')
            out['m2_yoy'] = pd.to_numeric(out['m2_yoy'], errors='coerce')
            out = out.dropna(subset=['date', 'm2_yoy'])
            out = out[out['date'] >= '2020-01-01']
            out = out.sort_values('date').drop_duplicates(subset=['date'], keep='last')
            out['date'] = out['date'].dt.strftime('%Y-%m')
            out = out.drop_duplicates(subset=['date'], keep='last')

            out.to_csv(save_path, index=False, encoding="utf-8")
            return True, f"shape=({len(out)}, {len(out.columns)})"

        except Exception as e:
            last_error = str(e)
            time.sleep(0.8)

    # 回退缓存
    if os.path.exists(save_path):
        try:
            cached = pd.read_csv(save_path)
            if len(cached) > 0 and {'date', 'm2_yoy'}.issubset(set(cached.columns)):
                return True, f"shape=({len(cached)}, {len(cached.columns)}) [cached]"
        except Exception:
            pass

    return False, last_error



In [11]:
print("开始下载宏观经济数据...")
print("="*60)

# 下载CPI数据
print("\n正在下载: CPI同比增速...")
success, msg = download_cpi_data(macro_dir)
if success:
    write_log(log_file, "SUCCESS", "macro_cpi", msg)
else:
    write_log(log_file, "FAILED", "macro_cpi", msg)

time.sleep(0.5)

# 下载M2数据
print("\n正在下载: M2同比增速...")
success, msg = download_m2_data(macro_dir)
if success:
    write_log(log_file, "SUCCESS", "macro_m2", msg)
else:
    write_log(log_file, "FAILED", "macro_m2", msg)

print("\n" + "="*60)
print("宏观数据下载完成！")
print("\n宏观数据选择理由：")
print("- CPI同比增速：反映通货膨胀水平，影响货币政策和股票估值")
print("- M2同比增速：反映货币供应量变化，与股市流动性密切相关")

开始下载宏观经济数据...

正在下载: CPI同比增速...


[2026-04-08 16:57:26] SUCCESS  macro_cpi  shape=(68, 2)



正在下载: M2同比增速...


[2026-04-08 16:57:32] SUCCESS  macro_m2  shape=(68, 2)

宏观数据下载完成！

宏观数据选择理由：
- CPI同比增速：反映通货膨胀水平，影响货币政策和股票估值
- M2同比增速：反映货币供应量变化，与股市流动性密切相关


## 6. 下载财务指标数据

In [12]:
def extract_finance_long_from_abstract(df, code):
    """将 stock_financial_abstract 宽表提取为长格式（每只股票最新可得5个年度）"""
    indicator_map = {
        "净资产收益率(ROE)": "ROE",
        "销售净利率": "净利润率"
    }

    # 仅保留“常用指标”分组，避免同名指标重复
    df = df[df['选项'].astype(str) == '常用指标'].copy()

    annual_cols = sorted(
        [c for c in df.columns if str(c).isdigit() and str(c).endswith('1231')],
        reverse=True
    )
    latest_annual_cols = annual_cols[:5]

    rows = []
    for raw_name, std_name in indicator_map.items():
        sub = df[df['指标'].astype(str) == raw_name]
        if sub.empty:
            continue

        s = sub.iloc[0]
        for col in latest_annual_cols:
            year = int(str(col)[:4])
            value = pd.to_numeric(s[col], errors='coerce')
            if pd.notna(value):
                rows.append({
                    'code': code,
                    'year': year,
                    'indicator': std_name,
                    'value': float(value)
                })

    return rows



In [13]:
print("开始下载财务指标数据...")
print("="*60)

all_finance_data = []
finance_cache_path = os.path.join(finance_dir, "finance_ratios.csv")

for stock in stock_list:
    code = stock["code"]
    name = stock["name"]

    print(f"\n正在获取: {name} ({code})...")

    success = False
    last_error = "No data returned"

    # 在线获取重试
    for _ in range(3):
        try:
            df = _run_without_proxy(ak.stock_financial_abstract, symbol=code)

            if df is None or len(df) == 0:
                last_error = "No data returned"
                continue

            rows = extract_finance_long_from_abstract(df, code)
            if len(rows) > 0:
                all_finance_data.extend(rows)
                write_log(log_file, "SUCCESS", f"finance_{code}", f"records={len(rows)}")
                print(f"  提取到 {len(rows)} 条（最新可得5年 × 2个指标）")
                success = True
                break
            else:
                last_error = "No valid rows extracted"

        except Exception as e:
            last_error = str(e)
            time.sleep(0.8)

    # 在线失败时回退已有缓存（按股票代码筛选）
    if not success and os.path.exists(finance_cache_path):
        try:
            cached = pd.read_csv(finance_cache_path, dtype={'code': str})
            cached['code'] = cached['code'].astype(str).str.zfill(6)
            cached_rows = cached[cached['code'] == code][['code', 'year', 'indicator', 'value']]
            if len(cached_rows) > 0:
                all_finance_data.extend(cached_rows.to_dict('records'))
                write_log(log_file, "SUCCESS", f"finance_{code}", f"records={len(cached_rows)} [cached]")
                print(f"  使用缓存 {len(cached_rows)} 条")
                success = True
        except Exception:
            pass

    if not success:
        write_log(log_file, "FAILED", f"finance_{code}", last_error)
        print(f"  获取失败: {last_error}")

print("\n" + "="*60)
print("财务数据下载完成！")



开始下载财务指标数据...

正在获取: 招商银行 (600036)...


[2026-04-08 16:57:35] SUCCESS  finance_600036  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 工商银行 (601398)...


[2026-04-08 16:57:37] SUCCESS  finance_601398  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 比亚迪 (002594)...


[2026-04-08 16:57:38] SUCCESS  finance_002594  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 上汽集团 (600104)...


[2026-04-08 16:57:39] SUCCESS  finance_600104  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 万科A (000002)...


...

[2026-04-08 16:57:41] SUCCESS  finance_600519  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 五粮液 (000858)...


[2026-04-08 16:57:42] SUCCESS  finance_000858  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 中国石油 (601857)...


[2026-04-08 16:57:43] SUCCESS  finance_601857  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 中兴通讯 (000063)...


[2026-04-08 16:57:46] SUCCESS  finance_000063  records=10
  提取到 10 条（最新可得5年 × 2个指标）

正在获取: 顺丰控股 (002352)...


[2026-04-08 16:57:46] SUCCESS  finance_002352  records=10
  提取到 10 条（最新可得5年 × 2个指标）

财务数据下载完成！


In [14]:
# 汇总并保存财务长格式数据
finance_df = pd.DataFrame(all_finance_data)

if len(finance_df) > 0:
    # 去重：同一股票同一年同指标只保留一条
    finance_df = finance_df.drop_duplicates(['code', 'year', 'indicator']).copy()
    finance_df = finance_df.sort_values(['code', 'year', 'indicator'])

    save_path = os.path.join(finance_dir, "finance_ratios.csv")
    finance_df.to_csv(save_path, index=False, encoding='utf-8')

    write_log(log_file, "SUCCESS", "finance_ratios", f"records={len(finance_df)}")

    print(f"财务数据已保存: {save_path}")
    print(f"总记录数: {len(finance_df)}")
    print(f"覆盖年度: {sorted(finance_df['year'].unique())}")
    print(f"指标类型: {finance_df['indicator'].unique().tolist()}")

    display(finance_df.head(20))
else:
    write_log(log_file, "FAILED", "finance_ratios", "No valid records extracted")
    print("未提取到有效财务数据")



[2026-04-08 16:57:46] SUCCESS  finance_ratios  records=100
财务数据已保存: data/finance/finance_ratios.csv
总记录数: 100
覆盖年度: [2020, 2021, 2022, 2023, 2024, 2025]
指标类型: ['ROE', '净利润率']


,code,year,indicator,value
44,000002,2021,ROE,9.780000
49,000002,2021,净利润率,8.407622
43,000002,2022,ROE,9.480000
48,000002,2022,净利润率,7.452967
42,000002,2023,ROE,4.910000
47,000002,2023,净利润率,4.392064
41,000002,2024,ROE,-24.410000
46,000002,2024,净利润率,-14.192097
40,000002,2025,ROE,-55.420000
45,000002,2025,净利润率,-39.330421


## 7. 验证下载数据

In [15]:
# 验证股票数据
print("股票数据验证：")
print("-" * 40)
for stock in stock_list:
    code = stock["code"]
    file_path = os.path.join(stock_dir, f"stock_{code}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{stock['name']:8s} ({code}): {len(df)} 行, {len(df.columns)} 列")
    else:
        print(f"{stock['name']:8s} ({code}): 文件不存在")

股票数据验证：
----------------------------------------
招商银行     (600036): 1516 行, 7 列
工商银行     (601398): 1516 行, 7 列
比亚迪      (002594): 1516 行, 7 列
上汽集团     (600104): 1516 行, 7 列
万科A      (000002): 1515 行, 7 列
贵州茅台     (600519): 1515 行, 7 列
五粮液      (000858): 1515 行, 7 列
中国石油     (601857): 1515 行, 7 列
中兴通讯     (000063): 1514 行, 7 列
顺丰控股     (002352): 1512 行, 7 列


In [16]:
# 验证指数数据
print("\n指数数据验证：")
print("-" * 40)
for idx in index_list:
    code = idx["code"]
    file_path = os.path.join(index_dir, f"index_{code}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{idx['name']:8s} ({code}): {len(df)} 行")
    else:
        print(f"{idx['name']:8s} ({code}): 文件不存在")


指数数据验证：
----------------------------------------
沪深300    (000300): 1515 行
中证500    (000905): 1515 行


In [17]:
# 验证宏观数据
print("\n宏观数据验证：")
print("-" * 40)
for macro_name in ["cpi", "m2"]:
    file_path = os.path.join(macro_dir, f"macro_{macro_name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{macro_name.upper():8s}: {len(df)} 行")
    else:
        print(f"{macro_name.upper():8s}: 文件不存在")


宏观数据验证：
----------------------------------------
CPI     : 68 行
M2      : 68 行


In [18]:
# 查看下载日志
print("\n下载日志：")
print("-" * 60)
with open(log_file, "r", encoding="utf-8") as f:
    print(f.read())


下载日志：
------------------------------------------------------------
[2026-04-08 16:56:38] SUCCESS  stock_600036  shape=(1516, 7) [cached]
[2026-04-08 16:56:42] SUCCESS  stock_601398  shape=(1516, 7) [cached]
[2026-04-08 16:56:48] SUCCESS  stock_002594  shape=(1516, 7) [cached]
[2026-04-08 16:56:51] SUCCESS  stock_600104  shape=(1516, 7) [cached]
[2026-04-08 16:56:54] SUCCESS  stock_000002  shape=(1515, 7) [cached]
[2026-04-08 16:56:58] SUCCESS  stock_600519  shape=(1515, 7) [cached]
[2026-04-08 16:57:02] SUCCESS  stock_000858  shape=(1515, 7) [cached]
[2026-04-08 16:57:09] SUCCESS  stock_601857  shape=(1515, 7) [cached]
[2026-04-08 16:57:13] SUCCESS  stock_000063  shape=(1514, 7) [cached]
[2026-04-08 16:57:17] SUCCESS  stock_002352  shape=(1512, 7) [cached]
[2026-04-08 16:57:18] SUCCESS  index_000300  shape=(1515, 6) [sina]
[2026-04-08 16:57:19] SUCCESS  index_000905  shape=(1515, 6) [sina]
[2026-04-08 16:57:26] SUCCESS  macro_cpi  shape=(68, 2)
[2026-04-08 16:57:32] SUCCESS  macro_m2 

## 8. 保存股票列表信息

In [19]:
# 保存股票列表信息为CSV，便于后续使用
stock_info_df = pd.DataFrame(stock_list)
stock_info_df['select_reason'] = [
    "股份制银行龙头，零售业务领先",
    "国有大行代表，市值最大",
    "新能源汽车领军企业",
    "传统汽车制造龙头",
    "房地产开发龙头企业",
    "白酒行业龙头，高端消费代表",
    "浓香型白酒龙头",
    "能源行业龙头，油气开采",
    "通信设备龙头企业",
    "快递物流龙头企业"
]
stock_info_df.to_csv(os.path.join(data_dir, "stock_info.csv"), index=False, encoding="utf-8")
print("股票信息已保存")
display(stock_info_df)

股票信息已保存


,code,name,industry,select_reason
0,600036,招商银行,银行,股份制银行龙头，零售业务领先
1,601398,工商银行,银行,国有大行代表，市值最大
2,002594,比亚迪,汽车,新能源汽车领军企业
3,600104,上汽集团,汽车,传统汽车制造龙头
4,000002,万科A,房地产,房地产开发龙头企业
5,600519,贵州茅台,白酒,白酒行业龙头，高端消费代表
6,000858,五粮液,白酒,浓香型白酒龙头
7,601857,中国石油,能源,能源行业龙头，油气开采
8,000063,中兴通讯,通讯,通信设备龙头企业
9,002352,顺丰控股,物流,快递物流龙头企业


In [20]:
print("\n数据下载完成！")
print("="*60)
print("下一步：运行 02_clean.ipynb 进行数据清洗")


数据下载完成！
下一步：运行 02_clean.ipynb 进行数据清洗
